# Notebook 02: MPC Basics — Mô hình xe và Iterative MPC

**Mục tiêu học tập:**
- Hiểu kinematic bicycle model
- Hiểu cách MPC tối ưu hóa control signal
- Demo MPC bám quỹ đạo đơn giản (đường tròn, đường sin)

**Tài liệu tham khảo:**
- [CVXPY Documentation](https://www.cvxpy.org/)
- [mpc_python Repository](https://github.com/mcarfagno/mpc_python)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from controller.vehicle_model import VehicleModel
from controller.iterative_mpc import IterativeMPC

## 1. Kinematic Bicycle Model

Mô hình động học xe đạp:
```
x_{t+1}   = x_t + v_t * cos(yaw_t) * dt
y_{t+1}   = y_t + v_t * sin(yaw_t) * dt
yaw_{t+1} = yaw_t + (v_t / L) * tan(steer_t) * dt
v_{t+1}   = v_t + accel_t * dt
```

- `state`: [x, y, yaw, v]
- `control`: [steer, accel]

In [ ]:
# Test vehicle model: forward, turn, accelerate
vm = VehicleModel(dt=0.1, wheelbase=0.3)

# Forward motion
state = np.array([0.0, 0.0, 0.0, 1.0])
states_fwd = [state.copy()]
for _ in range(50):
    state = vm.step(state, np.array([0.0, 0.0]))
    states_fwd.append(state.copy())

# Turning motion
state = np.array([0.0, 0.0, 0.0, 1.0])
states_turn = [state.copy()]
for _ in range(50):
    state = vm.step(state, np.array([0.3, 0.0]))  # steer = 0.3 rad
    states_turn.append(state.copy())

states_fwd = np.array(states_fwd)
states_turn = np.array(states_turn)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(states_fwd[:, 0], states_fwd[:, 1], 'b-', label='Forward (steer=0)')
ax.plot(states_turn[:, 0], states_turn[:, 1], 'r-', label='Turn (steer=0.3)')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Vehicle Model: Forward vs Turn')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

## 2. MPC — Bám quỹ đạo đường tròn

In [ ]:
# Generate circular reference trajectory
dt = 0.1
R = 3.0  # radius
omega = 0.5  # angular velocity
T = 2 * np.pi / omega  # one full circle
n_steps = int(T / dt)
t = np.linspace(0, T, n_steps + 1)

ref_circle = np.zeros((n_steps + 1, 4))
ref_circle[:, 0] = R * np.cos(omega * t)  # x
ref_circle[:, 1] = R * np.sin(omega * t)  # y
ref_circle[:, 2] = omega * t + np.pi / 2   # yaw (tangent)
ref_circle[:, 3] = R * omega               # v

# Run MPC
config = {'dt': 0.1, 'horizon': 15, 'Q': [1.0, 1.0, 0.5, 0.1],
          'R': [0.1, 0.1], 'max_steer': 0.5, 'max_accel': 2.0,
          'max_speed': 3.0, 'vehicle_length': 0.3, 'num_iterations': 3}
mpc = IterativeMPC(config)

# Simulate
state = ref_circle[0].copy()
actual = [state.copy()]
controls = []
for k in range(n_steps):
    ref_window = ref_circle[k:k + mpc.N + 1]
    control, status = mpc.solve(state, ref_window)
    state = mpc.vehicle.step(state, control)
    actual.append(state.copy())
    controls.append(control)

actual = np.array(actual)
controls = np.array(controls)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(ref_circle[:, 0], ref_circle[:, 1], 'b--', label='Reference', linewidth=2)
ax.plot(actual[:, 0], actual[:, 1], 'r-', label='MPC', linewidth=1.5)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('MPC: Circular Trajectory Tracking')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

ax = axes[1]
pos_err = np.sqrt((actual[:, 0] - ref_circle[:, 0])**2 + (actual[:, 1] - ref_circle[:, 1])**2)
ax.plot(t, pos_err, 'g-')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Position Error (m)')
ax.set_title('Tracking Error')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Position RMSE: {np.sqrt(np.mean(pos_err**2)):.4f} m")

## 3. MPC — Bám quỹ đạo đường sin

In [ ]:
# Sinusoidal reference
dt = 0.1
n_steps = 200
t = np.arange(n_steps + 1) * dt

ref_sin = np.zeros((n_steps + 1, 4))
ref_sin[:, 0] = t * 1.0                              # x: linear
ref_sin[:, 1] = 1.0 * np.sin(0.5 * t)                # y: sinusoidal
ref_sin[:, 2] = np.arctan2(0.5 * np.cos(0.5 * t), 1.0)  # yaw
ref_sin[:, 3] = np.sqrt(1.0**2 + (0.5 * np.cos(0.5 * t))**2)  # v

# Run MPC
state = ref_sin[0].copy()
actual = [state.copy()]
for k in range(n_steps):
    ref_window = ref_sin[k:k + mpc.N + 1]
    control, _ = mpc.solve(state, ref_window)
    state = mpc.vehicle.step(state, control)
    actual.append(state.copy())

actual = np.array(actual)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ref_sin[:, 0], ref_sin[:, 1], 'b--', label='Reference', linewidth=2)
ax.plot(actual[:, 0], actual[:, 1], 'r-', label='MPC', linewidth=1.5)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('MPC: Sinusoidal Trajectory Tracking')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Ảnh hưởng của Prediction Horizon

Thử các horizon khác nhau và quan sát ảnh hưởng đến RMSE.

In [ ]:
horizons = [5, 10, 15, 20]
results = []

for h in horizons:
    cfg = dict(config)
    cfg['horizon'] = h
    mpc_h = IterativeMPC(cfg)
    
    state = ref_circle[0].copy()
    errs = []
    for k in range(n_steps):
        ref_w = ref_circle[k:k + mpc_h.N + 1]
        ctrl, _ = mpc_h.solve(state, ref_w)
        state = mpc_h.vehicle.step(state, ctrl)
        e = np.sqrt((state[0] - ref_circle[k+1, 0])**2 + (state[1] - ref_circle[k+1, 1])**2)
        errs.append(e)
    results.append({'horizon': h, 'rmse': np.sqrt(np.mean(np.array(errs)**2))})

for r in results:
    print(f"Horizon {r['horizon']:>2}: RMSE = {r['rmse']:.4f} m")

## Tóm tắt

- **Kinematic bicycle model** mô tả chuyển động xe qua 4 state: x, y, yaw, v
- **MPC** tối ưu control bằng cách giải QP lặp (iterative linearization + CVXPY)
- Prediction horizon dài hơn thường cho RMSE thấp hơn nhưng tốn computation

**Bài tập:**
1. Thay đổi Q và R, quan sát ảnh hưởng đến tracking và smoothness
2. Tăng `num_iterations`, kiểm tra MPC hội tụ tốt hơn không